In [ ]:
def optimize_milk_items(milk_df, meal_name='Meal', max_items=5, nutrient='Calcium', calorie_limit=None, binary=True):
    """
    Optimizes selection of milk items for a meal (e.g., breakfast or lunch)
    to maximize a specific nutrient, subject to a max number of items and optional calorie cap.
    
    Parameters:
        milk_df (pd.DataFrame): Milk items for the meal
        meal_name (str): Label for the meal (for printouts)
        max_items (int): Max number of items to select
        nutrient (str): Nutrient to maximize (e.g. 'Calcium')
        calorie_limit (float or None): Max total calories (optional)
        binary (bool): If True, selects items as 0 or 1 only
        
    Returns:
        selected_items (pd.DataFrame): Selected milk items with quantities
    """
    from scipy.optimize import linprog
    import numpy as np

    print(f"\n--- Optimizing Milk Items for {meal_name} ---")
    print(f"Objective: Maximize '{nutrient}' with up to {max_items} items\n")

    milk_df = milk_df.copy()
    milk_df.columns = milk_df.columns.str.strip()
    
    # Handle missing nutrient or calorie values
    nutrient_vals = milk_df[nutrient].fillna(0).values
    calories = milk_df['Calories'].fillna(0).values
    n = len(milk_df)

    # Objective: maximize nutrient (linprog minimizes, so negate)
    c = -nutrient_vals

    # Total item constraint
    A_eq = [np.ones(n)]
    b_eq = [max_items]

    # Optional calorie constraint
    A_ub = []
    b_ub = []
    if calorie_limit is not None:
        A_ub.append(calories)
        b_ub.append(calorie_limit)

    # Bounds: binary (0 or 1), or allow fractional quantities
    bounds = [(0, 1 if binary else None) for _ in range(n)]

    # Solve
    result = linprog(
        c=c,
        A_eq=A_eq,
        b_eq=b_eq,
        A_ub=A_ub if A_ub else None,
        b_ub=b_ub if b_ub else None,
        bounds=bounds,
        method='highs'
    )

    if result.success:
        milk_df['Quantity'] = result.x.round(0 if binary else 2)
        selected = milk_df[milk_df['Quantity'] > 0].copy()

        print(f"✅ Optimization successful for {meal_name}")
        print(f"Maximized {nutrient}: {selected[nutrient].dot(selected['Quantity']):.2f}")
        if calorie_limit:
            total_calories = selected['Calories'].dot(selected['Quantity'])
            print(f"Total Calories Used: {total_calories:.2f} / {calorie_limit}")
        
        print(f"\nSelected {len(selected)} items for {meal_name}:")
        print(selected[['name', 'Quantity', nutrient, 'Calories']])

        return selected
    else:
        print(f"❌ Optimization failed for {meal_name}: {result.message}")
        return None


In [ ]:
# Run breakfast optimization
milk_breakfast_selected = optimize_milk_items(
    milk_df=milk_b,
    meal_name='Breakfast',
    max_items=4,
    nutrient='Calcium (mg)',
    calorie_limit=600,
    binary=True
)

# Run lunch optimization
milk_lunch_selected = optimize_milk_items(
    milk_df=milk_l,
    meal_name='Lunch',
    max_items=5,
    nutrient='Calcium (mg)',
    calorie_limit=700,
    binary=True
)
